In [2]:
import pandas as pd
import numpy as np

# DATA CLEANING

In [3]:
# 1. Đọc dữ liệu thô
file_path = '../datas/binh_duong_real_estates.csv'
df = pd.read_csv(file_path)
initial_rows, initial_cols = df.shape
print(f"=== BẮT ĐẦU TIẾN TRÌNH LÀM SẠCH DỮ LIỆU (Ban đầu: {initial_rows:,} dòng | {initial_cols} cột) ===\n")

# 2. [R1] Loại bỏ cột khuyết > 80%
drop_cols = ['house_depth', 'floor_count', 'road_width', 'balcony_direction']
existing_drop_cols = [c for c in drop_cols if c in df.columns]
df_clean = df.drop(columns=existing_drop_cols)
print(f"[R1] Đã loại bỏ {len(existing_drop_cols)} cột khuyết nặng > 80%.")

# 3. [R2 & R5] Xóa dòng khuyết price và area < 10m²
df_clean = df_clean.dropna(subset=['price'])
df_clean = df_clean[df_clean['area'] >= 10]
print(f"[R2 & R5] Đã xóa toàn bộ dòng khuyết 'price' và các dòng có 'area' < 10 m².")

# 4. [R6] Chuẩn hóa khoảng trắng chuỗi văn bản (Trim/Strip)
text_cols = ['district_name', 'ward_name', 'name']
for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()
print(f"[R6] Đã xóa khoảng trắng thừa ở đầu/cuối chuỗi văn bản.")

# 5. [R3] Gán giá trị mặc định cho thuộc tính Dimension khuyết
impute_cols = {
    'ward_name': 'Chưa xác định',
    'house_direction': 'Chưa xác định',
    'legal_status': 'Chưa xác định'
}
for col, val in impute_cols.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(val)
print(f"[R3] Đã gán giá trị mặc định 'Chưa xác định' cho các thuộc tính Dimension.")

# 6. [R4] Khử trùng lặp nghiệp vụ (Business Deduplication)
biz_cols = ['name', 'price', 'area', 'ward_name', 'district_name']
valid_biz_cols = [c for c in biz_cols if c in df_clean.columns]
df_clean = df_clean.drop_duplicates(subset=valid_biz_cols, keep='first')
print(f"[R4] Đã lọc trùng lặp nghiệp vụ dựa trên bộ Khóa nghiệp vụ chuẩn.")

# 7. [R7] Xử lý dữ liệu ngoại lệ cực đoan (Outliers & Anomalies)
# Lọc bỏ các dòng tổng giá cực đoan > 500 tỷ VNĐ ra khỏi dataset chính
df_outliers = df_clean[df_clean['price'] > 500_000_000_000]
df_clean = df_clean[df_clean['price'] <= 500_000_000_000]
print(f"[R7] Đã lọc bỏ {len(df_outliers)} dòng có tổng giá cực đoan (> 500 tỷ VNĐ) sang vùng lưu trữ riêng.")

# 8. Xuất file dữ liệu sạch
final_rows, final_cols = df_clean.shape
output_clean_path = '../datas/cleaned_binh_duong_real_estates.csv'
output_outlier_path = '../datas/outliers_binh_duong_real_estates.csv'

df_clean.to_csv(output_clean_path, index=False, encoding='utf-8-sig')
df_outliers.to_csv(output_outlier_path, index=False, encoding='utf-8-sig')

print(f"\n=== HOÀN THÀNH TIẾN TRÌNH LÀM SẠCH ===")
print(f"- Số dòng sạch còn lại: {final_rows:,} dòng (Đã lọc bỏ {initial_rows - final_rows:,} dòng rác/ngoại lệ)")
print(f"- Số cột còn lại: {final_cols} cột")
print(f"- File dữ liệu sạch đã lưu tại: {output_clean_path}")
print(f"- File dữ liệu ngoại lệ đã lưu tại: {output_outlier_path}")

=== BẮT ĐẦU TIẾN TRÌNH LÀM SẠCH DỮ LIỆU (Ban đầu: 42,783 dòng | 19 cột) ===

[R1] Đã loại bỏ 4 cột khuyết nặng > 80%.
[R2 & R5] Đã xóa toàn bộ dòng khuyết 'price' và các dòng có 'area' < 10 m².
[R6] Đã xóa khoảng trắng thừa ở đầu/cuối chuỗi văn bản.
[R3] Đã gán giá trị mặc định 'Chưa xác định' cho các thuộc tính Dimension.
[R4] Đã lọc trùng lặp nghiệp vụ dựa trên bộ Khóa nghiệp vụ chuẩn.
[R7] Đã lọc bỏ 17 dòng có tổng giá cực đoan (> 500 tỷ VNĐ) sang vùng lưu trữ riêng.

=== HOÀN THÀNH TIẾN TRÌNH LÀM SẠCH ===
- Số dòng sạch còn lại: 36,303 dòng (Đã lọc bỏ 6,480 dòng rác/ngoại lệ)
- Số cột còn lại: 15 cột
- File dữ liệu sạch đã lưu tại: ../datas/cleaned_binh_duong_real_estates.csv
- File dữ liệu ngoại lệ đã lưu tại: ../datas/outliers_binh_duong_real_estates.csv


In [4]:
# 1. Khởi tạo đường dẫn file sạch
clean_path = '../datas/cleaned_binh_duong_real_estates.csv'
outlier_path = '../datas/outliers_binh_duong_real_estates.csv'

print("=" * 70)
print("BÁO CÁO NGHIỆM THU ĐỐI SOÁT DỮ LIỆU SẠCH (POST-ETL CHECK)")
print("=" * 70)

try:
    df_clean = pd.read_csv(clean_path)
    print(f"[OK] Đã tải file sạch thành công: {df_clean.shape[0]:,} dòng | {df_clean.shape[1]} cột\n")
except FileNotFoundError:
    print(f"[ERROR] Không tìm thấy file tại đường dẫn: {clean_path}")
    exit()

# 2. Kiểm tra R1: Cấu trúc cột (Đã drop 4 cột khuyết > 80% chưa?)
dropped_cols = ['house_depth', 'floor_count', 'road_width', 'balcony_direction']
r1_check = any(col in df_clean.columns for col in dropped_cols)
print(f"1. [R1] Kiểm tra loại bỏ 4 cột khuyết > 80%: {'THẤT BẠI (Vẫn còn cột rác)' if r1_check else 'ĐẠT (Đã loại bỏ trọn vẹn 4 cột)'}")

# 3. Kiểm tra R2: Khuyết giá bất động sản (price)
r2_null_price = df_clean['price'].isnull().sum()
print(f"2. [R2] Kiểm tra dòng khuyết 'price': {'ĐẠT (0 dòng NULL)' if r2_null_price == 0 else f'THẤT BẠI (Còn {r2_null_price} dòng NULL)'}")

# 4. Kiểm tra R5: Ranh giới diện tích (area < 10m2)
r5_invalid_area = (df_clean['area'] < 10).sum()
print(f"3. [R5] Kiểm tra ranh giới 'area' < 10m²: {'ĐẠT (0 dòng vi phạm)' if r5_invalid_area == 0 else f'THẤT BẠI (Còn {r5_invalid_area} dòng area < 10)'}")

# 5. Kiểm tra R6: Khoảng trắng thừa (Trim/Strip)
text_cols = ['district_name', 'ward_name', 'name']
r6_spaces = 0
for col in text_cols:
    if col in df_clean.columns:
        r6_spaces += (df_clean[col].astype(str) != df_clean[col].astype(str).str.strip()).sum()
print(f"4. [R6] Kiểm tra khoảng trắng thừa đầu/cuối văn bản: {'ĐẠT (0 dòng dính khoảng trắng)' if r6_spaces == 0 else f'THẤT BẠI (Còn {r6_spaces} dòng chưa strip)'}")

# 6. Kiểm tra R3: Gán giá trị mặc định 'Chưa xác định'
r3_null_ward = df_clean['ward_name'].isnull().sum()
unidentified_ward_count = (df_clean['ward_name'] == 'Chưa xác định').sum()
print(f"5. [R3] Kiểm tra gán mặc định 'ward_name': {'ĐẠT (0 dòng NULL)' if r3_null_ward == 0 else 'THẤT BẠI'} | Số dòng 'Chưa xác định': {unidentified_ward_count:,}")

# 7. Kiểm tra R4: Trùng lặp nghiệp vụ (Business Duplicates)
biz_cols = ['name', 'price', 'area', 'ward_name', 'district_name']
r4_duplicates = df_clean.duplicated(subset=biz_cols).sum()
print(f"6. [R4] Kiểm tra trùng lặp nghiệp vụ theo Business Key: {'ĐẠT (0 dòng trùng)' if r4_duplicates == 0 else f'THẤT BẠI (Còn {r4_duplicates} dòng trùng)'}")

# 8. Kiểm tra R7: Ngoại lệ tổng giá cực đoan (> 500 tỷ VNĐ)
r7_extreme_price = (df_clean['price'] > 500_000_000_000).sum()
print(f"7. [R7] Kiểm tra tổng giá cực đoan > 500 tỷ VNĐ: {'ĐẠT (0 dòng vi phạm)' if r7_extreme_price == 0 else f'THẤT BẠI (Còn {r7_extreme_price} dòng > 500 tỷ)'}")

# 9. Kiểm tra file Staging cô lập Outliers
try:
    df_outliers = pd.read_csv(outlier_path)
    print(f"\n[STAGING CHECK] File cô lập ngoại lệ ({outlier_path}): Đã lưu trữ an toàn {len(df_outliers)} dòng.")
except FileNotFoundError:
    print(f"\n[STAGING CHECK] Không tìm thấy file ngoại lệ tại: {outlier_path}")

print("=" * 70)
print("KẾT LUẬN: Tập dữ liệu sạch đạt chuẩn 100% sẵn sàng đưa vào SSIS / SQL Server.")
print("=" * 70)

BÁO CÁO NGHIỆM THU ĐỐI SOÁT DỮ LIỆU SẠCH (POST-ETL CHECK)
[OK] Đã tải file sạch thành công: 36,303 dòng | 15 cột

1. [R1] Kiểm tra loại bỏ 4 cột khuyết > 80%: ĐẠT (Đã loại bỏ trọn vẹn 4 cột)
2. [R2] Kiểm tra dòng khuyết 'price': ĐẠT (0 dòng NULL)
3. [R5] Kiểm tra ranh giới 'area' < 10m²: ĐẠT (0 dòng vi phạm)
4. [R6] Kiểm tra khoảng trắng thừa đầu/cuối văn bản: ĐẠT (0 dòng dính khoảng trắng)
5. [R3] Kiểm tra gán mặc định 'ward_name': ĐẠT (0 dòng NULL) | Số dòng 'Chưa xác định': 3,170
6. [R4] Kiểm tra trùng lặp nghiệp vụ theo Business Key: ĐẠT (0 dòng trùng)
7. [R7] Kiểm tra tổng giá cực đoan > 500 tỷ VNĐ: ĐẠT (0 dòng vi phạm)

[STAGING CHECK] File cô lập ngoại lệ (../datas/outliers_binh_duong_real_estates.csv): Đã lưu trữ an toàn 17 dòng.
KẾT LUẬN: Tập dữ liệu sạch đạt chuẩn 100% sẵn sàng đưa vào SSIS / SQL Server.
